# Reproduce the whole project

Runs every stage of the MMA3001 apple-tree scaffold project in order, in Google Colab.

**Before you start:** put the two dataset zip files in a Google Drive folder
`MyDrive/MMA3001_data` (the dataset is not in this repository). If the repository is
private, add a GitHub token to Colab Secrets (key icon) as `GH_PAT`.

Then choose **Runtime > Run all**. Everything is seeded, so scores come out the same as
in `results/`; only run times differ.

| Step | Script | Time (approx.) |
|---|---|---|
| Set-up, tests | | 3 min |
| 1 Data audit | `scripts/audit_dataset.py` | 2 min |
| 1b Mask alignment check | `scripts/alignment_check.py` | 15 min |
| 2 Depth-hole filling | `scripts/hole_filling_experiment.py` | 2 min |
| 3 Per-pixel classifiers vs baselines | `scripts/scaffold_experiment.py` | 10 min |
| 4a Shape feasibility | `scripts/shape_feasibility.py` | 2 min |
| 4b Shape method, validation ladder | `scripts/shape_method.py` | 8 min |
| 4c Settings and performance | `scripts/shape_settings.py` | 15 min |
| 4d Final test | `scripts/final_test.py` | 8 min |
| 4e All methods side by side | `scripts/method_gallery.py` | 8 min |
| Documentation | `pdoc` | < 1 min |

## Set-up: Drive, repository, packages, data

In [ ]:
import os, glob, zipfile
from google.colab import drive

REPO_URL = "https://github.com/JieliLeong/MMA3001_AppleTree_Project.git"
DRIVE_DATA = "/content/drive/MyDrive/MMA3001_data"    # folder holding the dataset zips
REPO_DIR, DATA_DIR = "/content/MMA3001_AppleTree_Project", "/content/data"

drive.mount("/content/drive")
url = REPO_URL
try:                                                  # private repository: token from Secrets
    from google.colab import userdata
    url = REPO_URL.replace("https://", f"https://oauth2:{userdata.get('GH_PAT')}@")
except Exception:
    pass
if not os.path.isdir(REPO_DIR):
    !git clone -q {url} {REPO_DIR}
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt

if not glob.glob(f"{DATA_DIR}/**/*.png", recursive=True):
    for path in sorted(glob.glob(f"{DRIVE_DATA}/*.zip")):
        print("unzipping", os.path.basename(path))
        with zipfile.ZipFile(path) as archive:
            archive.extractall(DATA_DIR)
print("PNG files:", len(glob.glob(f"{DATA_DIR}/**/*.png", recursive=True)))

## Tests (67)

In [ ]:
!python -m pytest -q

## 1. Data audit
Completeness of every tree's files, image properties, the train / validation / test split by tree, and ground-truth overlays.

In [ ]:
!python scripts/audit_dataset.py {DATA_DIR} --out results/audit --sample 30 --overlays 6

## 1b. Do the supplied masks line up with the images?

Shifts and stretches every mask over its summer and winter photo and depth image (about 15 minutes).

In [ ]:
!python scripts/alignment_check.py {DATA_DIR} --out results/alignment

## 2. Depth-hole filling
Laplace solvers (direct, Jacobi, Gauss-Seidel, SOR) vs interpolation on holes with known answers (training trees only).

In [ ]:
!python scripts/hole_filling_experiment.py {DATA_DIR} --out results/holes --frames 20

## 3. Per-pixel classifiers vs baselines
Trained on training trees, thresholds tuned on validation trees, scored on test trees.

In [ ]:
!python scripts/scaffold_experiment.py {DATA_DIR} --out results/scaffold

## 4a. Can the scaffold be predicted as a shape?
Representation quality, SVD modes and oracle limits (training and validation trees only).

In [ ]:
!python scripts/shape_feasibility.py {DATA_DIR} --out results/shape_check

## 4b. Shape method: validation ladder
Ridge, KNN, random forest and MLP regressors for the shape modes; Monte Carlo probability band.

In [ ]:
!python scripts/shape_method.py {DATA_DIR} --out results/shape_method

## 4c. Settings and performance
Settings chosen by rules fixed in advance; Monte Carlo convergence, learning curve, seeds, timing, FLOPs, memory.

In [ ]:
!python scripts/shape_settings.py {DATA_DIR} --out results/shape_settings

## 4d. Final test
The chosen configuration retrained on training + validation trees, scored once on the test trees.

In [ ]:
!python scripts/final_test.py {DATA_DIR} --out results/final --save-masks

## 4e. Every method side by side

Refits the Stage 3 pixel models exactly as before and draws every method's mask on the same three test trees (about 8 minutes).

In [ ]:
!python scripts/method_gallery.py {DATA_DIR} --out results/gallery

## Documentation

In [ ]:
!pdoc --math ./apple_scaffold -o ./docs
print(sorted(os.listdir("docs")))